# Solar filaments: native mask ensemble + instance proposals + learned PQ selection

**Target: improve on the 0.33 public run. `pq50` is an experiment name and a 0.50 goal,
not a measured score or a guarantee.** Run all cells on Kaggle **GPU T4 ×2**, with the
competition data attached and Internet enabled for packages and generic pretrained weights.

The saved YOLO11 run scored **0.4234 held-out PQ**, with **473 TP / 211 FP / 329 FN**;
public PQ was 0.33. Under-500-pixel calibration recall was only **23.4%**. Disk-fit
failures had **0.0927 report PQ**, although that cohort is small and confounded by
image difficulty. Extending the same YOLO training is not the main experiment here.

This notebook trains a **YOLO11m box detector at 2048**, a **ResNet34 U-Net on native
512 crops**, and an **EfficientNetV2-S U-Net on native 768 crops**. The mask ensemble
predicts foreground and contours on overlapping native tiles. Detection boxes keep
instance identity; semantic components can recover detector misses. A small learned
quality regressor estimates whether a mask earns official PQ credit. Calibration
compares hybrid and semantic-only output, learned and simple selection, three pixel
thresholds and two minimum sizes. An adaptive candidate mode lets learned quality
choose different mask thresholds for different instances, with nested-duplicate
suppression. It does not assume hybrid fusion is always better.

Two GPUs train with DDP; each rank gets **two loader workers**, four total. Test
inference shards images between two GPU processes. Preprocessing and candidate
evaluation use four bounded CPU workers. Exact flips/90° rotations preserve crop
detail; full disks are not resized for the mask network. Detector resizing is only
used in the explicitly reduced smoke mode.

The split unit is the **physical observation date**, before annotator expansion.
Training samples one annotator view at a time and oversamples small instances.
It never makes an all-annotator hard union. All images receive photometric
normalization; uncertain solar geometry only disables radial correction.

Checkpoints are selected with a fixed-rule calibration PQ every four epochs;
postprocessing does **not** run after each training batch. Candidate-quality scores
are crossfitted by calibration date. The report set is excluded from all fitting
and recipe selection. Calibration remains a **selection score**, because it also
selects checkpoints. The independent report measures the selected pre-refit
pipeline. With `refit_full=True`, the frozen schedule then retrains on all supplied
training annotations; the earlier report score is **not** a score for that refit.

Default full mode performs selection training **and full-data refitting**. This is
a substantial training run, not a five-epoch quick fix. Runtime on two T4s has not
been benchmarked. Epoch logs show time, optimizer updates and VRAM. Completed
stages and prediction files are reused within the same fingerprinted run directory.
Use `mode="smoke"` first to test your environment; its small CSV is explicitly
named `smoke_submission.csv` and must not be submitted.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import subprocess
import sys

# Keep Kaggle's CUDA-enabled torch/torchvision installation. Do not replace it.
requirements = {
    'ultralytics': ('ultralytics', '8.4.198'),
    'segmentation_models_pytorch': ('segmentation-models-pytorch', '0.5.0'),
    'timm': ('timm', '1.0.30'),
    'scikit-learn': ('scikit-learn', '1.9.1'),
}
install = []
for distribution, (package, version) in requirements.items():
    try:
        installed = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install.append(f'{package}=={version}')
for package in ['pycocotools', 'scipy', 'opencv-python-headless', 'pandas', 'matplotlib', 'joblib', 'pyyaml']:
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        install.append(package)
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *install])
print('Dependencies ready. If these libraries were imported earlier in this kernel, restart it before proceeding.')

In [ ]:
from pathlib import Path
RUNTIME_PATH = Path.cwd() / 'pq50_runtime.py'
RUNTIME_SOURCE = '"""Train-only, date-grouped detector + native mask learning for MAGFiLO Kaggle.\n\nEmbedded verbatim by build_pq50_notebook.py. No external filament annotations,\nprecomputed submissions, or test-derived supervision are used.\n"""\nfrom __future__ import annotations\n\nimport os\nfor _key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):\n    os.environ[_key] = "1"\nimport argparse\nimport copy\nimport csv\nimport gc\nimport hashlib\nimport json\nimport math\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom collections import defaultdict, deque\nfrom concurrent.futures import ThreadPoolExecutor\nfrom contextlib import nullcontext\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom pycocotools import mask as mu\nfrom scipy import ndimage as ndi\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset, DistributedSampler\n\ncv2.setNumThreads(1)\ntorch.set_num_threads(1)\nSIZE = 2048\nVERSION = "pq50-native-hybrid-v1"\n\n\ndef default_config():\n    return dict(mode="full", data_root="", output_root="pq50_outputs", run_name="native_hybrid_v1",\n        gpu_ids=[0, 1], cpu_workers=4, seed=20261103, refit_full=True,\n        calibration_fraction=.12, report_fraction=.12, minimum_free_gib=2.,\n        dense_models=[dict(encoder="resnet34", crop=512),\n                      dict(encoder="tu-tf_efficientnetv2_s", crop=768)],\n        pretrained=True, dense_epochs=32, dense_steps=288, dense_batch_per_gpu=2,\n        dense_lr=3e-4, weight_decay=1e-4, accumulation=3, ema_decay=.995,\n        pq_every=4, tile_size=1024, tile_overlap=256, inference_batch=2,\n        tta=4, detector_weights="yolo11m.pt", detector_size=2048,\n        detector_epochs=45, detector_batch_per_gpu=1, detector_lr=.001,\n        detector_conf=.07, max_det=100, max_dense_candidates=512, quality_folds=5,\n        pixel_thresholds=[.35, .5, .65], smoke_sources=3)\n\n\ndef save_json(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + ".tmp")\n    temp.write_text(json.dumps(data, indent=2, allow_nan=False), encoding="utf8")\n    os.replace(temp, path)\n\n\ndef load_json(path):\n    return json.loads(Path(path).read_text(encoding="utf8"))\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef file_sha(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef bounded_map(fn, items, workers=4):\n    """At most four active/pending native images; never queue the entire dataset."""\n    items = iter(items)\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        pending = deque()\n        for _ in range(workers):\n            item = next(items, None)\n            if item is not None:\n                pending.append(pool.submit(fn, item))\n        while pending:\n            yield pending.popleft().result()\n            item = next(items, None)\n            if item is not None:\n                pending.append(pool.submit(fn, item))\n\n\ndef encode(mask):\n    rle = mu.encode(np.asfortranarray(mask, dtype=np.uint8))\n    return dict(size=[int(x) for x in rle["size"]], counts=rle["counts"].decode("ascii"))\n\n\ndef decode(rle):\n    rle = dict(rle)\n    if isinstance(rle["counts"], str):\n        rle["counts"] = rle["counts"].encode("ascii")\n    return mu.decode(rle).astype(bool)\n\n\ndef annotation_rle(ann, shape=(SIZE, SIZE)):\n    seg = ann["segmentation"]\n    if isinstance(seg, list):\n        r = mu.merge(mu.frPyObjects(seg, *shape))\n    elif isinstance(seg["counts"], list):\n        r = mu.frPyObjects(seg, *shape)\n    else:\n        r = dict(seg)\n        if isinstance(r["counts"], str):\n            r["counts"] = r["counts"].encode("ascii")\n    return dict(size=list(shape), counts=r["counts"].decode("ascii"))\n\n\ndef read_gray(path):\n    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n    if image is None or image.shape != (SIZE, SIZE):\n        raise ValueError(f"Expected a readable native {SIZE}x{SIZE} image: {path}")\n    return image\n\n\ndef normalize_solar(gray):\n    """Always normalize. Uncertain geometry disables radial correction only.\n\n    Three uint8 channels: global standardization, bounded radial correction,\n    CLAHE of the corrected view. No hard limb clipping or morphology inpainting.\n    """\n    if gray.ndim != 2 or gray.dtype != np.uint8:\n        raise ValueError("uint8 grayscale required")\n    h, w = gray.shape\n    small = cv2.resize(gray, (256, 256), interpolation=cv2.INTER_AREA)\n    threshold, binary = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    good, cx, cy, radius = False, w / 2, h / 2, .45 * min(h, w)\n    if contours:\n        contour = max(contours, key=cv2.contourArea)\n        if len(contour) >= 5:\n            (x, y), (a, b), _ = cv2.fitEllipse(contour)\n            good = (100 < x < 156 and 100 < y < 156 and 180 < min(a, b) < 253\n                    and max(a, b) / max(min(a, b), 1) < 1.13)\n            if good:\n                cx, cy = x * w / 256, y * h / 256\n                radius = (a + b) * min(h, w) / (4 * 256)\n    yy, xx = np.ogrid[:h, :w]\n    rr = np.sqrt(((xx - cx) / radius)**2 + ((yy - cy) / radius)**2)\n    support = (rr < .88) & (gray > max(5, threshold * .35))\n    if support.sum() < h * w * .1:\n        support = (rr < .75)\n    values = gray[support].astype(np.float32)\n    mean, std = float(values.mean()), max(float(values.std()), 8.)\n    raw = np.clip(128 + 32 * (gray.astype(np.float32) - mean) / std, 0, 255)\n    corrected = gray.astype(np.float32)\n    if good:\n        # Statistics are estimated at 256² to avoid repeated full-disk sorting.\n        sr = cv2.resize(rr.astype(np.float32), (256, 256), interpolation=cv2.INTER_AREA)\n        bins = np.minimum((sr * 48).astype(int), 63)\n        profile = np.array([np.median(small[(bins == k) & (small > 5)])\n                            if np.any((bins == k) & (small > 5)) else np.nan for k in range(48)])\n        valid = np.flatnonzero(np.isfinite(profile))\n        if len(valid) > 24:\n            profile = np.interp(np.arange(48), valid, profile[valid])\n            profile = ndi.gaussian_filter1d(profile, 2)\n            reference = float(np.median(profile[5:35]))\n            gain = np.clip(reference / np.maximum(profile, 20), .75, 1.35)\n            correction = np.interp(np.minimum(rr, 47 / 48) * 48, np.arange(48), gain)\n            corrected *= correction.astype(np.float32)\n    cvalues = corrected[support]\n    corrected = np.clip(128 + 32 * (corrected - float(cvalues.mean())) /\n                        max(float(cvalues.std()), 8.), 0, 255).astype(np.uint8)\n    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(16, 16)).apply(corrected)\n    return np.stack([raw.astype(np.uint8), corrected, clahe], -1), dict(\n        geometry_ok=bool(good), mean=mean, std=std)\n\n\ndef discover_root(configured=""):\n    name = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"\n    candidates = [Path(configured)] if configured else []\n    if not configured:\n        for parent in [Path.cwd(), *Path.cwd().parents]:\n            candidates.extend([parent, parent / "MAGFilo_1.0_Kaggle_2026"])\n        if Path("/kaggle/input").exists():\n            candidates.extend(p.parent.parent for p in Path("/kaggle/input").rglob(name))\n    for root in candidates:\n        if (root / "train" / name).is_file() and (root / "test" / "test_images").is_dir():\n            return root.resolve()\n    raise FileNotFoundError("Attach the competition data or set CFG[\'data_root\'] to its train/test parent.")\n\n\ndef index_data(root, cfg):\n    path = root / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"\n    data = load_json(path)\n    annotations = defaultdict(list)\n    for a in data["annotations"]:\n        annotations[a["image_id"]].append(a)\n    by_name = defaultdict(list)\n    for row in data["images"]:\n        assert (row["height"], row["width"]) == (SIZE, SIZE)\n        rles = [annotation_rle(a) for a in annotations[row["id"]]]\n        boxes = [list(map(float, mu.toBbox({**r, "counts": r["counts"].encode()}))) for r in rles]\n        by_name[Path(row["file_name"]).name].append(dict(id=row["id"], rles=rles, boxes=boxes))\n    sources = [dict(name=n, stem=Path(n).stem, date=Path(n).stem[:8], views=v,\n                    image=str(root / "train" / "train_images" / n)) for n, v in sorted(by_name.items())]\n    dates = np.array(sorted({s["date"] for s in sources}))\n    np.random.default_rng(cfg["seed"]).shuffle(dates)\n    nr = max(1, round(len(dates) * cfg["report_fraction"]))\n    nc = max(1, round(len(dates) * cfg["calibration_fraction"]))\n    report, calibration = set(dates[:nr]), set(dates[nr:nr + nc])\n    splits = {"train": [], "calibration": [], "report": []}\n    for source in sources:\n        key = "report" if source["date"] in report else "calibration" if source["date"] in calibration else "train"\n        splits[key].append(source)\n    tests = [dict(name=p.name, stem=p.stem, date=p.stem[:8], image=str(p), views=[])\n             for p in sorted((root / "test" / "test_images").glob("*"))\n             if p.suffix.lower() in (".jpg", ".jpeg", ".png")]\n    if {s["stem"] for s in sources} & {s["stem"] for s in tests}:\n        raise ValueError("Train/test filenames overlap.")\n    if cfg["mode"] == "smoke":\n        splits = {k: v[:cfg["smoke_sources"]] for k, v in splits.items()}\n        tests = tests[:cfg["smoke_sources"]]\n    splits["test"] = tests\n    assert not (set(s["date"] for s in splits["train"]) &\n                set(s["date"] for k in ("calibration", "report") for s in splits[k]))\n    return splits, file_sha(path)\n\n\ndef prepare(cfg):\n    cfg = copy.deepcopy(cfg)\n    for computed in ("run", "fingerprint", "annotation_sha", "runtime_sha"):\n        cfg.pop(computed, None)\n    if cfg["cpu_workers"] != 4:\n        raise ValueError("This run reserves exactly four CPU workers in total.")\n    if cfg["mode"] not in ("full", "smoke"):\n        raise ValueError("mode must be full or smoke")\n    if cfg["mode"] == "smoke":\n        cfg.update(dense_epochs=1, dense_steps=2, pq_every=1, detector_epochs=1,\n                   detector_size=512, tta=1, refit_full=False, tile_size=512,\n                   tile_overlap=64, dense_batch_per_gpu=1, accumulation=1,\n                   max_dense_candidates=32, pixel_thresholds=[.5])\n        cfg["dense_models"] = [dict(encoder="resnet18", crop=256)]\n        cfg["detector_weights"] = "yolo11n.pt"\n    root = discover_root(cfg["data_root"])\n    cfg["data_root"] = str(root)\n    cfg["output_root"] = str(Path(cfg["output_root"]).resolve())\n    splits, annotation_sha = index_data(root, cfg)\n    image_hashes = dict(bounded_map(lambda s: (s["stem"], file_sha(s["image"])),\n        [s for rows in splits.values() for s in rows], cfg["cpu_workers"]))\n    runtime_sha = file_sha(__file__)\n    fingerprint = digest(dict(version=VERSION, cfg=cfg, annotations=annotation_sha,\n                              sources={k: [s["name"] for s in v] for k, v in splits.items()},\n                              images=image_hashes, runtime=runtime_sha))\n    run = Path(cfg["output_root"]).resolve() / (cfg["run_name"] + "_" + fingerprint[:10])\n    run.mkdir(parents=True, exist_ok=True)\n    cfg.update(run=str(run), fingerprint=fingerprint, annotation_sha=annotation_sha, runtime_sha=runtime_sha)\n    all_sources = [s for rows in splits.values() for s in rows]\n    # Measure actual compressed cache size, rather than budgeting multiple RGB copies.\n    cache = run / "cache"\n    cache.mkdir(exist_ok=True)\n    sample_sizes = []\n    for s in all_sources[:min(4, len(all_sources))]:\n        feature, _ = normalize_solar(read_gray(s["image"]))\n        ok, png = cv2.imencode(".png", feature, [cv2.IMWRITE_PNG_COMPRESSION, 3])\n        if not ok:\n            raise OSError("PNG encoder failed")\n        sample_sizes.append(len(png))\n    # Only missing files need disk headroom on rerun. Targets are small bit-packed PNGs.\n    missing = sum(not (cache / (s["stem"] + ".png")).exists() for s in all_sources)\n    estimated = (max(sample_sizes) * missing * 1.35 + 200_000 *\n                 sum(len(s["views"]) for s in all_sources)) / 2**30\n    free = shutil.disk_usage(run).free / 2**30\n    required = estimated + 3.5 + cfg["minimum_free_gib"]\n    print(f"Disk free {free:.1f} GiB; estimated new cache {estimated:.1f}, models/maps 3.5, "\n          f"reserve {cfg[\'minimum_free_gib\']:.1f}. No Ultralytics disk cache or duplicated images.", flush=True)\n    if free < required:\n        raise OSError(f"Estimated incremental need {required:.1f} GiB, free {free:.1f}. "\n                      "Choose an output disk with more room; no dataset is deleted automatically.")\n\n    def process(s):\n        s = copy.deepcopy(s)\n        s["image_sha"] = image_hashes[s["stem"]]\n        target = cache / (s["stem"] + ".png")\n        meta_path = cache / (s["stem"] + ".json")\n        if not target.is_file() or not meta_path.is_file():\n            image, meta = normalize_solar(read_gray(s["image"]))\n            if not cv2.imwrite(str(target), image, [cv2.IMWRITE_PNG_COMPRESSION, 3]):\n                raise OSError(target)\n            save_json(meta_path, meta)\n        s.update(cache=str(target), photometry=load_json(meta_path))\n        for view in s["views"]:\n            packed_path = cache / (view["id"] + "_target.png")\n            if not packed_path.is_file():\n                fg, edges = np.zeros((SIZE, SIZE), np.uint8), np.zeros((SIZE, SIZE), np.uint8)\n                for rle in view["rles"]:\n                    mask = decode(rle).astype(np.uint8)\n                    fg |= mask\n                    edges |= cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8))\n                if not cv2.imwrite(str(packed_path), fg + 2 * edges):\n                    raise OSError(packed_path)\n            view["target"] = str(packed_path)\n        return s\n\n    for name, sources in splits.items():\n        splits[name] = list(bounded_map(process, sources, cfg["cpu_workers"]))\n        print(f"Prepared {name}: {len(sources)} physical images, "\n              f"{sum(len(s[\'views\']) for s in sources)} annotator records", flush=True)\n    import importlib.metadata as metadata\n    packages = {name: metadata.version(name) for name in\n                ("torch", "torchvision", "ultralytics", "segmentation-models-pytorch", "timm", "scikit-learn", "pycocotools")}\n    manifest = dict(config=cfg, splits=splits, packages=packages, python=sys.version)\n    save_json(run / "manifest.json", manifest)\n    save_json(run / "config.json", cfg)\n    prepare_detector(manifest)\n    return cfg, manifest\n\n\ndef prepare_detector(manifest):\n    """Original per-annotator boxes; hard-link images so records cost no RGB copies."""\n    cfg, splits = manifest["config"], manifest["splits"]\n    root = Path(cfg["run"]) / "detector_data"\n    for part in ("train", "calibration", "report"):\n        (root / "images" / part).mkdir(parents=True, exist_ok=True)\n        (root / "labels" / part).mkdir(parents=True, exist_ok=True)\n        for source in splits[part]:\n            for view in source["views"]:\n                target = root / "images" / part / (view["id"] + ".png")\n                if not target.exists():\n                    try:\n                        os.link(source["cache"], target)\n                    except OSError:\n                        # Same output filesystem normally supports hardlinks. Do not silently copy GBs.\n                        os.symlink(source["cache"], target)\n                lines = []\n                for x, y, w, h in view["boxes"]:\n                    lines.append(f"0 {(x + w/2)/SIZE:.9f} {(y + h/2)/SIZE:.9f} {w/SIZE:.9f} {h/SIZE:.9f}")\n                (root / "labels" / part / (view["id"] + ".txt")).write_text("\\n".join(lines), encoding="utf8")\n    train_files = sorted((root / "images" / "train").glob("*.png"))\n    all_files = train_files + sorted((root / "images" / "calibration").glob("*.png")) + sorted((root / "images" / "report").glob("*.png"))\n    for name, files in (("all", all_files), ("train", train_files)):\n        (root / (name + ".txt")).write_text("\\n".join(str(p.absolute()) for p in files), encoding="utf8")\n    for full in (False, True):\n        data = dict(path=str(root), train=str(root / ("all.txt" if full else "train.txt")),\n                    val=str(root / "images" / ("train" if full else "calibration")), names={0: "filament"})\n        import yaml\n        (root / ("full.yaml" if full else "selection.yaml")).write_text(yaml.safe_dump(data), encoding="utf8")\n\n\ndef device_setup(cfg, distributed=False):\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required for training. Use Kaggle GPU T4 x2.")\n    local = int(os.environ.get("LOCAL_RANK", 0))\n    gpu = cfg["gpu_ids"][local]\n    torch.cuda.set_device(gpu)\n    if distributed and len(cfg["gpu_ids"]) > 1:\n        dist.init_process_group("gloo" if os.name == "nt" else "nccl", timeout=timedelta(minutes=45))\n    return torch.device(f"cuda:{gpu}"), local, len(cfg["gpu_ids"])\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef build_dense(spec, pretrained=False):\n    import segmentation_models_pytorch as smp\n    model = smp.Unet(encoder_name=spec["encoder"], encoder_weights="imagenet" if pretrained else None,\n        in_channels=3, classes=2, decoder_channels=(128, 64, 32, 16, 16), decoder_attention_type="scse")\n    # Stable statistics at small per-GPU batch. Encoder ImageNet BN statistics stay frozen.\n    def replace_bn(module):\n        for name, child in list(module.named_children()):\n            if isinstance(child, nn.BatchNorm2d):\n                setattr(module, name, nn.GroupNorm(math.gcd(8, child.num_features), child.num_features))\n            else:\n                replace_bn(child)\n    replace_bn(model.decoder)\n    # SMP\'s last decoder block has no skip input, so attention1 is never called.\n    # Exclude those six dormant SCSE parameters from DDP gradient reduction.\n    model.decoder.blocks[-1].attention1.requires_grad_(False)\n    return model\n\n\ndef train_mode(model):\n    model.train()\n    for child in model.encoder.modules():\n        if isinstance(child, nn.modules.batchnorm._BatchNorm):\n            child.eval()\n\n\nclass CropDataset(Dataset):\n    def __init__(self, sources, crop, length, seed):\n        self.sources, self.crop, self.length, self.seed, self.epoch = sources, crop, length, seed, 0\n        self.cache = {}  # Each loader worker retains only its last physical image.\n\n    def __len__(self):\n        return self.length\n\n    def __getitem__(self, index):\n        rng = np.random.default_rng(self.seed + self.epoch * 10_000_019 + int(index))\n        s = self.sources[int(rng.integers(len(self.sources)))]\n        view = s["views"][int(rng.integers(len(s["views"])))]\n        key = view["id"]\n        if key not in self.cache:\n            self.cache.clear()\n            self.cache[key] = (cv2.imread(s["cache"]), cv2.imread(view["target"], 0))\n        image, packed = self.cache[key]\n        size = self.crop\n        choice = rng.random()\n        if choice < .65 and view["boxes"]:\n            boxes = np.array(view["boxes"])\n            weights = np.maximum(boxes[:, 2] * boxes[:, 3], 50)**(-.3)\n            x, y, w, h = boxes[rng.choice(len(boxes), p=weights / weights.sum())]\n            cx, cy = x + rng.random() * w, y + rng.random() * h\n            x0 = round(cx - size * rng.uniform(.25, .75))\n            y0 = round(cy - size * rng.uniform(.25, .75))\n        elif choice < .85:\n            # Genuine dark-region backgrounds, including sunspots and imperfect disks.\n            points = rng.integers(150, SIZE - 150, size=(24, 2))\n            j = np.argmin(image[points[:, 0], points[:, 1], 1])\n            y0, x0 = points[j] - size // 2\n        else:\n            x0, y0 = rng.integers(0, SIZE - size + 1, size=2)\n        x0, y0 = int(np.clip(x0, 0, SIZE - size)), int(np.clip(y0, 0, SIZE - size))\n        x = image[y0:y0 + size, x0:x0 + size].copy()\n        p = packed[y0:y0 + size, x0:x0 + size]\n        y = np.stack([(p & 1) > 0, (p & 2) > 0], -1).astype(np.float32)\n        k = int(rng.integers(4))\n        x, y = np.rot90(x, k), np.rot90(y, k)\n        if rng.random() < .5:\n            x, y = x[:, ::-1], y[:, ::-1]\n        x = x.astype(np.float32) / 255\n        x = np.clip((x - .5) * rng.uniform(.8, 1.2) + .5 + rng.uniform(-.06, .06), 0, 1)\n        x = (x - np.array([.485, .456, .406], np.float32)) / np.array([.229, .224, .225], np.float32)\n        return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))), torch.from_numpy(np.ascontiguousarray(y.transpose(2, 0, 1)))\n\n\ndef dense_loss(logits, targets):\n    with torch.autocast(device_type=logits.device.type, enabled=False):\n        logits, targets = logits.float(), targets.float()\n        p = logits.sigmoid()\n        fg, edge = targets[:, :1], targets[:, 1:2]\n        # Local contour errors receive extra weight without inflating every positive pixel.\n        weights = 1 + 3 * edge\n        bce = (F.binary_cross_entropy_with_logits(logits[:, :1], fg, reduction="none") * weights).mean()\n        dims = (1, 2, 3)\n        dice = 1 - ((2 * (p[:, :1] * fg).sum(dims) + 1) /\n                    (p[:, :1].sum(dims) + fg.sum(dims) + 1)).mean()\n        edge_bce = F.binary_cross_entropy_with_logits(logits[:, 1:2], edge, pos_weight=logits.new_tensor(3.))\n        return .45 * bce + .45 * dice + .1 * edge_bce\n\n\ndef update_ema(ema, model, decay):\n    with torch.no_grad():\n        for dst, src in zip(ema.parameters(), model.parameters()):\n            dst.lerp_(src.detach(), 1 - decay)\n        for dst, src in zip(ema.buffers(), model.buffers()):\n            dst.copy_(src.detach())\n\n\ndef tile_starts(size, tile, overlap):\n    if tile >= size:\n        return [0]\n    return sorted(set([*range(0, size - tile + 1, tile - overlap), size - tile]))\n\n\n@torch.inference_mode()\ndef dense_predict(model, image, device, tile=1024, overlap=256, batch=2, tta=4):\n    h, w = image.shape[:2]\n    tile = min(tile, h, w)\n    if tile % 32 or not 0 <= overlap < tile or tta not in (1, 4, 8):\n        raise ValueError("Tiles must be divisible by 32; 0 <= overlap < tile; TTA = 1/4/8")\n    positions = [(y, x) for y in tile_starts(h, tile, overlap) for x in tile_starts(w, tile, overlap)]\n    hann = np.maximum(np.outer(np.hanning(tile), np.hanning(tile)), .02).astype(np.float32)\n    total, weight = np.zeros((2, h, w), np.float32), np.zeros((h, w), np.float32)\n    model.eval()\n    for start in range(0, len(positions), batch):\n        pos = positions[start:start + batch]\n        values = np.stack([image[y:y + tile, x:x + tile] for y, x in pos]).astype(np.float32) / 255\n        values = (values - np.array([.485, .456, .406])) / np.array([.229, .224, .225])\n        inp = torch.from_numpy(values.transpose(0, 3, 1, 2).copy()).to(device, dtype=torch.float32)\n        probabilities = torch.zeros((len(pos), 2, tile, tile), device=device, dtype=torch.float32)\n        for t in range(tta):\n            dims = [] if t % 4 == 0 else [-1] if t % 4 == 1 else [-2] if t % 4 == 2 else [-2, -1]\n            z = inp.transpose(-1, -2) if t >= 4 else inp\n            z = z.flip(dims) if dims else z\n            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                p = model(z).float().sigmoid()\n            p = p.flip(dims) if dims else p\n            p = p.transpose(-1, -2) if t >= 4 else p\n            probabilities.add_(p, alpha=1 / tta)\n        arrays = probabilities.cpu().numpy()\n        for (y, x), p in zip(pos, arrays):\n            total[:, y:y + tile, x:x + tile] += p * hann\n            weight[y:y + tile, x:x + tile] += hann\n    return total / np.maximum(weight, 1e-6)\n\n\ndef metric_counts(gt, predicted):\n    if gt and predicted:\n        a = [{**r, "counts": r["counts"].encode()} for r in gt]\n        b = [{**r, "counts": r["counts"].encode()} for r in predicted]\n        iou = np.asarray(mu.iou(a, b, [0] * len(b)))\n    else:\n        iou = np.zeros((len(gt), len(predicted)))\n    hit = iou > .5\n    return np.array([iou[hit].sum(), hit.sum(), (~hit.any(0)).sum(), (~hit.any(1)).sum()], np.float64)\n\n\ndef summarize(counts):\n    iou, tp, fp, fn = map(float, counts)\n    denominator = tp + .5 * (fp + fn)\n    return dict(PQ=iou / denominator if denominator else 0., SQ=iou / tp if tp else 0.,\n                RQ=tp / denominator if denominator else 0., TP=int(tp), FP=int(fp), FN=int(fn), sum_IoU=iou)\n\n\ndef components(prob, threshold=.5, minimum=20, maximum=None):\n    n, labels, stats, _ = cv2.connectedComponentsWithStats((prob >= threshold).astype(np.uint8), 8)\n    ids = np.flatnonzero(stats[1:, cv2.CC_STAT_AREA] >= minimum) + 1\n    if maximum is not None and len(ids) > maximum:\n        sums = np.bincount(labels.ravel(), weights=prob.ravel(), minlength=n)\n        means = sums[ids] / stats[ids, cv2.CC_STAT_AREA]\n        ids = ids[np.argsort(-means, kind="stable")[:maximum]]\n    for k in ids:\n        x, y, w, h, area = map(int, stats[k])\n        if area >= minimum:\n            yield (x, y, x + w, y + h), labels[y:y + h, x:x + w] == k\n\n\ndef full_rle(box, mask, shape=None):\n    shape = (SIZE, SIZE) if shape is None else shape\n    x0, y0, x1, y1 = box\n    if mask.shape != (y1 - y0, x1 - x0):\n        raise ValueError("ROI and mask shapes differ")\n    if mask.size > shape[0] * shape[1] // 8:\n        image = np.zeros(shape, np.uint8)\n        image[y0:y1, x0:x1] = mask\n        return encode(image)\n    # Encode sparse ROIs directly in the GLOBAL column-major index space. Avoid\n    # allocating/encoding a 4-million-pixel zero canvas for every tiny candidate.\n    cols, rows = np.nonzero(mask.T)\n    indices = (cols.astype(np.int64) + x0) * shape[0] + rows + y0\n    if not len(indices):\n        counts = [shape[0] * shape[1]]\n    else:\n        breaks = np.flatnonzero(np.diff(indices) != 1)\n        starts = indices[np.r_[0, breaks + 1]]\n        ends = indices[np.r_[breaks, len(indices) - 1]]\n        runs = np.empty(len(starts) * 2, np.int64)\n        runs[0::2] = starts - np.r_[-1, ends[:-1]] - 1\n        runs[1::2] = ends - starts + 1\n        counts = runs.tolist()\n        tail = shape[0] * shape[1] - int(ends[-1]) - 1\n        if tail:\n            counts.append(tail)\n    rle = mu.frPyObjects(dict(size=list(shape), counts=counts), *shape)\n    return dict(size=list(shape), counts=rle["counts"].decode("ascii"))\n\n\ndef checkpoint_save(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(".tmp")\n    torch.save(value, temp)\n    os.replace(temp, path)\n\n\ndef train_dense(config_path, member, full=False):\n    cfg = load_json(config_path)\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    device, rank, world = device_setup(cfg, distributed=True)\n    seed_all(cfg["seed"] + member * 101)\n    spec = cfg["dense_models"][member]\n    out = Path(cfg["run"]) / ("full_models" if full else "selection_models") / f"dense_{member}"\n    out.mkdir(parents=True, exist_ok=True)\n    done = out / "complete.json"\n    if done.is_file():\n        print(f"Already complete: {out}", flush=True)\n        if dist.is_initialized(): dist.destroy_process_group()\n        return\n    epochs = cfg["dense_epochs"]\n    if full:\n        epochs = load_json(Path(cfg["run"]) / "selection_models" / f"dense_{member}" / "complete.json")["best_epoch"]\n    sources = manifest["splits"]["train"]\n    if full:\n        sources = sources + manifest["splits"]["calibration"] + manifest["splits"]["report"]\n    steps = math.ceil(cfg["dense_steps"] * len(sources) / max(len(manifest["splits"]["train"]), 1)) if full else cfg["dense_steps"]\n    length = steps * cfg["dense_batch_per_gpu"] * world\n    dataset = CropDataset(sources, spec["crop"], length, cfg["seed"] + member * 111)\n    sampler = DistributedSampler(dataset, world, rank, shuffle=True, seed=cfg["seed"])\n    loader = DataLoader(dataset, batch_size=cfg["dense_batch_per_gpu"], sampler=sampler,\n        num_workers=cfg["cpu_workers"] // world, pin_memory=True, persistent_workers=False,\n        prefetch_factor=2, drop_last=True)\n    # Download the generic ImageNet initializer once before other ranks open it.\n    if rank == 0:\n        model = build_dense(spec, cfg["pretrained"])\n    if dist.is_initialized(): dist.barrier()\n    if rank != 0:\n        model = build_dense(spec, cfg["pretrained"])\n    model = model.to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["dense_lr"], weight_decay=cfg["weight_decay"])\n    scaler = torch.amp.GradScaler("cuda")\n    begin, best, best_epoch, updates = 0, -1., epochs, 0\n    last = out / "last_training.pt"\n    if last.is_file():\n        state = torch.load(last, map_location="cpu", weights_only=False)\n        if state["fingerprint"] != cfg["fingerprint"]:\n            raise RuntimeError("Checkpoint configuration mismatch")\n        model.load_state_dict(state["model"])\n        ema.load_state_dict(state["ema"])\n        optimizer.load_state_dict(state["optimizer"])\n        scaler.load_state_dict(state["scaler"])\n        begin, best, best_epoch, updates = state["epoch"], state["best"], state["best_epoch"], state["updates"]\n    wrapped = DDP(model, device_ids=[device.index], broadcast_buffers=False) if world > 1 else model\n    if begin == 0:\n        ema.load_state_dict(model.state_dict())\n    print(f"Dense member {member}, rank {rank}/{world}: {device}, {len(sources)} sources, "\n          f"native {spec[\'crop\']} crops, {len(loader)} microbatches/epoch, "\n          f"{cfg[\'cpu_workers\']//world} loader workers", flush=True)\n    for epoch in range(begin, epochs):\n        started = time.time()\n        dataset.epoch = epoch\n        sampler.set_epoch(epoch)\n        train_mode(model)\n        total = 0.\n        optimizer.zero_grad(set_to_none=True)\n        for step, (x, y) in enumerate(loader):\n            # Entire planned schedule reaches its low-LR end; no patience truncation.\n            fraction = (epoch + step / len(loader)) / max(epochs, 1)\n            warm = min(1., (epoch * len(loader) + step + 1) / max(len(loader), 1))\n            lr = cfg["dense_lr"] * warm * (.05 + .95 * (1 + math.cos(math.pi * fraction)) / 2)\n            for group in optimizer.param_groups: group["lr"] = lr\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            accumulation = cfg["accumulation"]\n            last_group = step // accumulation == (len(loader) - 1) // accumulation\n            group_size = len(loader) % accumulation if last_group and len(loader) % accumulation else accumulation\n            update = (step + 1) % accumulation == 0 or step + 1 == len(loader)\n            context = wrapped.no_sync() if world > 1 and not update else nullcontext()\n            with context:\n                with torch.autocast("cuda"):\n                    loss = dense_loss(wrapped(x), y)\n                scaler.scale(loss / group_size).backward()\n            total += loss.detach().item()\n            if update:\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), 1.)\n                previous_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                if scaler.get_scale() >= previous_scale:\n                    updates += 1\n                    update_ema(ema, model, min(cfg["ema_decay"], (1 + updates) / (10 + updates)))\n            if step == 0:\n                audit = dict(rank=rank, world_size=world, gpu=str(device), gpu_name=torch.cuda.get_device_name(device),\n                    observed_batch=int(x.shape[0]), native_crop=list(x.shape[-2:]),\n                    loader_workers=loader.num_workers, total_loader_workers=loader.num_workers * world)\n                save_json(out / f"rank_{rank}_resources.json", audit)\n                print(json.dumps(audit), flush=True)\n            elif (step + 1) % 32 == 0:\n                print(f"Dense {member} rank {rank}: epoch {epoch+1}/{epochs}, step {step+1}/{len(loader)}, "\n                      f"loss {total/(step+1):.4f}, lr {lr:.6f}", flush=True)\n        counts = np.zeros(4, np.float64)\n        evaluate = not full and ((epoch + 1) % cfg["pq_every"] == 0 or epoch + 1 == epochs)\n        if evaluate:\n            for image_index, source in enumerate(manifest["splits"]["calibration"][rank::world]):\n                image = cv2.imread(source["cache"])\n                p = dense_predict(ema, image, device, cfg["tile_size"], cfg["tile_overlap"], cfg["inference_batch"], 1)[0]\n                rles = [full_rle(box, mask) for box, mask in components(p, .5, 100) if p[box[1]:box[3], box[0]:box[2]][mask].max() > .8]\n                for view in source["views"]:\n                    counts += metric_counts(view["rles"], rles)\n                if (image_index + 1) % 10 == 0:\n                    print(f"Dense {member} rank {rank}: fixed-rule PQ check, {image_index+1} images", flush=True)\n        stats = torch.tensor([total / len(loader), *counts], device=device, dtype=torch.float64)\n        if world > 1: dist.all_reduce(stats)\n        values = stats.cpu().numpy()\n        pq = summarize(values[1:])["PQ"] if evaluate else None\n        if evaluate and pq > best:\n            best, best_epoch = pq, epoch + 1\n            if rank == 0:\n                checkpoint_save(out / "best.pt", dict(state=ema.state_dict(), spec=spec, epoch=best_epoch,\n                    fingerprint=cfg["fingerprint"], checkpoint_PQ=best))\n        if rank == 0:\n            checkpoint_save(out / "last.pt", dict(state=ema.state_dict(), spec=spec, epoch=epoch + 1,\n                fingerprint=cfg["fingerprint"]))\n            checkpoint_save(last, dict(model=model.state_dict(), ema=ema.state_dict(), optimizer=optimizer.state_dict(),\n                scaler=scaler.state_dict(), epoch=epoch + 1, best=best, best_epoch=best_epoch,\n                updates=updates, fingerprint=cfg["fingerprint"]))\n            log = dict(epoch=epoch + 1, loss=float(values[0] / world), fixed_rule_calibration_PQ=pq,\n                lr=lr, updates=updates, seconds=round(time.time() - started, 2),\n                peak_vram_gib=round(torch.cuda.max_memory_allocated(device) / 2**30, 3))\n            with (out / "history.jsonl").open("a", encoding="utf8") as f:\n                f.write(json.dumps(log) + "\\n")\n            print(json.dumps(log), flush=True)\n    if rank == 0:\n        if full:\n            shutil.copy2(out / "last.pt", out / "best.pt")\n        save_json(done, dict(best_epoch=best_epoch, fixed_rule_calibration_PQ=best if not full else None,\n            epochs=epochs, updates=updates, checkpoint_sha=file_sha(out / "best.pt")))\n        # Complete model weights remain; optimizer recovery state no longer needed.\n        last.unlink(missing_ok=True)\n    if world > 1:\n        dist.barrier()\n        dist.destroy_process_group()\n\n\ndef train_detector(config_path, full=False):\n    from ultralytics import YOLO, settings\n    from ultralytics.models.yolo.detect import DetectionTrainer\n    from ultralytics.data.build import build_dataloader\n    from ultralytics.utils.torch_utils import torch_distributed_zero_first\n    cfg = load_json(config_path)\n    rank = int(os.environ.get("LOCAL_RANK", 0))\n    world = len(cfg["gpu_ids"])\n    out = Path(cfg["run"]) / ("full_models" if full else "selection_models") / "detector"\n    if (out / "complete.json").is_file():\n        print(f"Already complete: {out}", flush=True)\n        return\n    settings.update({k: False for k in ("wandb", "mlflow", "clearml", "comet", "tensorboard", "raytune") if k in settings})\n\n    class FourWorkerTrainer(DetectionTrainer):\n        def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode="train"):\n            if mode == "val":\n                batch_size = 1\n            with torch_distributed_zero_first(rank):\n                dataset = self.build_dataset(dataset_path, mode, batch_size)\n            return build_dataloader(dataset, batch=batch_size,\n                workers=cfg["cpu_workers"] // world if mode == "train" else 0,\n                shuffle=mode == "train", rank=rank, drop_last=False, device=self.device)\n\n    epochs = cfg["detector_epochs"]\n    if full:\n        epochs = load_json(Path(cfg["run"]) / "selection_models" / "detector" / "complete.json")["best_epoch"]\n    source = out / "weights" / "last.pt"\n    resume = source.is_file()\n    model = YOLO(str(source) if resume else cfg["detector_weights"])\n    model.train(trainer=FourWorkerTrainer, data=str(Path(cfg["run"]) / "detector_data" /\n        ("full.yaml" if full else "selection.yaml")),\n        epochs=epochs, patience=0, imgsz=cfg["detector_size"],\n        batch=cfg["detector_batch_per_gpu"] * world, device=cfg["gpu_ids"],\n        workers=cfg["cpu_workers"] // world, optimizer="AdamW", lr0=cfg["detector_lr"],\n        lrf=.05, cos_lr=True, warmup_epochs=1., nbs=16, weight_decay=cfg["weight_decay"],\n        mosaic=0., mixup=0., copy_paste=0., degrees=0., translate=0., scale=0., shear=0.,\n        perspective=0., fliplr=.5, flipud=.5, hsv_h=0., hsv_s=0., hsv_v=.12,\n        cache=False, rect=False, multi_scale=False, amp=True, plots=False,\n        max_det=cfg["max_det"], seed=cfg["seed"], save_period=-1,\n        project=str(out.parent), name=out.name, exist_ok=True, resume=resume, verbose=False)\n    if rank == 0:\n        import pandas as pd\n        history = pd.read_csv(out / "results.csv")\n        history.columns = history.columns.str.strip()\n        field = "metrics/mAP50-95(B)"\n        best_epoch = int(history.loc[history[field].idxmax(), "epoch"]) if field in history else epochs\n        # Refit uses a fixed epoch count, never a training-set validation optimum.\n        selected = out / "weights" / ("last.pt" if full else "best.pt")\n        save_json(out / "complete.json", dict(best_epoch=epochs if full else best_epoch,\n            epochs=epochs, checkpoint=str(selected), checkpoint_sha=file_sha(selected)))\n\n\ndef command(config_path, task, *args, distributed=False):\n    cfg = load_json(config_path)\n    script = str(Path(__file__).resolve())\n    if distributed and len(cfg["gpu_ids"]) > 1:\n        cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone",\n               "--nnodes=1", f"--nproc_per_node={len(cfg[\'gpu_ids\'])}", script]\n    else:\n        cmd = [sys.executable, script]\n    cmd += ["--config", str(config_path), "--task", task, *map(str, args)]\n    print("Launching:", " ".join(cmd), flush=True)\n    # Stream progress into the notebook. A failed child fails its cell immediately.\n    subprocess.run(cmd, check=True)\n\n\ndef train_all(cfg, full=False):\n    path = Path(cfg["run"]) / "config.json"\n    flags = ["--full"] if full else []\n    selected = Path(cfg["run"]) / "selected_pipeline.json"\n    semantic_only = full and selected.is_file() and load_json(selected)["recipe"]["set"].startswith("dense_")\n    # Download a generic detector initializer in the parent, avoiding concurrent writes.\n    if not semantic_only:\n        from ultralytics import YOLO\n        detector = YOLO(cfg["detector_weights"])\n        del detector\n        command(path, "detector", *flags, distributed=True)\n    else:\n        print("Semantic-only recipe won: skipping the unused detector\'s full-data retraining.", flush=True)\n    for member in range(len(cfg["dense_models"])):\n        command(path, "dense", "--member", member, *flags, distributed=True)\n\n\ndef model_paths(cfg, full=False):\n    root = Path(cfg["run"]) / ("full_models" if full else "selection_models")\n    detector_info = root / "detector" / "complete.json"\n    if full and not detector_info.is_file():\n        selected = load_json(Path(cfg["run"]) / "selected_pipeline.json")\n        if not selected["recipe"]["set"].startswith("dense_"):\n            raise RuntimeError("Hybrid inference requires a completed full-data detector refit.")\n        detector_info = Path(cfg["run"]) / "selection_models" / "detector" / "complete.json"\n    detector = load_json(detector_info)["checkpoint"]\n    dense = [str(root / f"dense_{i}" / "best.pt") for i in range(len(cfg["dense_models"]))]\n    return dict(detector=detector, dense=dense)\n\n\ndef box_iou(box, boxes):\n    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)\n    inter = np.maximum(np.minimum(box[2:], boxes[:, 2:]) - np.maximum(box[:2], boxes[:, :2]), 0).prod(1)\n    a = np.maximum(np.array(box[2:]) - np.array(box[:2]), 0).prod()\n    b = np.maximum(boxes[:, 2:] - boxes[:, :2], 0).prod(1)\n    return inter / np.maximum(a + b - inter, 1e-9)\n\n\ndef fuse_boxes(rows, views):\n    """Cluster duplicates, retain independent instance proposals and TTA support."""\n    if not rows:\n        return np.zeros((0, 6), np.float32)\n    clusters = []\n    for row in sorted(rows, key=lambda x: x[4], reverse=True):\n        if clusters:\n            boxes = [np.average(np.array(c)[:, :4], axis=0, weights=np.array(c)[:, 4]) for c in clusters]\n            overlaps = box_iou(np.array(row[:4]), boxes)\n            index = int(overlaps.argmax())\n        else:\n            overlaps, index = np.zeros(0), -1\n        if index >= 0 and overlaps[index] >= .55:\n            clusters[index].append(row)\n        else:\n            clusters.append([row])\n    result = []\n    for c in clusters:\n        a = np.asarray(c)\n        box = np.average(a[:, :4], axis=0, weights=a[:, 4])\n        result.append([*box, float(a[:, 4].max()), len(set(a[:, 5].astype(int))) / views])\n    return np.asarray(result, np.float32)\n\n\n@torch.inference_mode()\ndef detector_predict(model, image, device, cfg):\n    rows = []\n    h, w = image.shape[:2]\n    # Four flips preserve the solar image\'s native sampling. TTA8 also transposes.\n    for t in range(cfg["tta"]):\n        z = image.transpose(1, 0, 2) if t >= 4 else image\n        flipx, flipy = t % 4 in (1, 3), t % 4 in (2, 3)\n        if flipx: z = z[:, ::-1]\n        if flipy: z = z[::-1]\n        result = model.predict(np.ascontiguousarray(z), imgsz=cfg["detector_size"], device=device,\n            conf=cfg["detector_conf"], iou=.55, max_det=cfg["max_det"], verbose=False, quantize=16)[0]\n        boxes = result.boxes.xyxy.cpu().numpy()\n        scores = result.boxes.conf.cpu().numpy()\n        for b, score in zip(boxes, scores):\n            x0, y0, x1, y1 = map(float, b)\n            if flipx: x0, x1 = w - x1, w - x0\n            if flipy: y0, y1 = h - y1, h - y0\n            if t >= 4: x0, y0, x1, y1 = y0, x0, y1, x1\n            rows.append([x0, y0, x1, y1, float(score), t])\n    return fuse_boxes(rows, cfg["tta"])\n\n\ndef prediction_key(cfg, paths):\n    return digest(dict(runtime=VERSION, run=cfg["fingerprint"],\n        models={k: [file_sha(p) for p in v] if isinstance(v, list) else file_sha(v) for k, v in paths.items()},\n        tile=cfg["tile_size"], overlap=cfg["tile_overlap"], tta=cfg["tta"], conf=cfg["detector_conf"]))[:16]\n\n\ndef predict_worker(config_path, part, gpu_slot=0, full=False):\n    from ultralytics import YOLO\n    cfg = load_json(config_path)\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    device = torch.device(f"cuda:{cfg[\'gpu_ids\'][gpu_slot]}")\n    torch.cuda.set_device(device)\n    paths = model_paths(cfg, full)\n    key = prediction_key(cfg, paths)\n    folder = Path(cfg["run"]) / "predictions" / key / part\n    folder.mkdir(parents=True, exist_ok=True)\n    models = []\n    for path in paths["dense"]:\n        checkpoint = torch.load(path, map_location="cpu", weights_only=False)\n        model = build_dense(checkpoint["spec"])\n        model.load_state_dict(checkpoint["state"])\n        models.append(model.to(device).eval())\n    detector = YOLO(paths["detector"])\n    sources = manifest["splits"][part][gpu_slot::len(cfg["gpu_ids"])]\n    for i, source in enumerate(sources):\n        target = folder / (source["stem"] + ".npz")\n        if target.is_file():\n            with np.load(target) as check:\n                if check["prob"].shape != (2, SIZE, SIZE) or not np.isfinite(check["boxes"]).all():\n                    raise ValueError(f"Corrupt prediction cache: {target}")\n            continue\n        image = cv2.imread(source["cache"])\n        probability = np.zeros((2, SIZE, SIZE), np.float32)\n        for model in models:\n            probability += dense_predict(model, image, device, cfg["tile_size"], cfg["tile_overlap"],\n                cfg["inference_batch"], cfg["tta"]) / len(models)\n        boxes = detector_predict(detector, image, device, cfg)\n        temp = target.with_suffix(".tmp")\n        with temp.open("wb") as f:\n            np.savez_compressed(f, prob=np.rint(np.clip(probability, 0, 1) * 255).astype(np.uint8), boxes=boxes)\n        os.replace(temp, target)\n        print(f"Prediction GPU {device.index}: {part} {i+1}/{len(sources)} {source[\'stem\']}, "\n              f"{len(boxes)} proposals", flush=True)\n\n\ndef predict_part(cfg, part, full=False):\n    paths = model_paths(cfg, full)\n    key = prediction_key(cfg, paths)\n    folder = Path(cfg["run"]) / "predictions" / key / part\n    flags = ["--full"] if full else []\n    # One process/model replica per GPU; disjoint image shards, two model pools total.\n    processes = []\n    try:\n        for slot in range(len(cfg["gpu_ids"])):\n            cmd = [sys.executable, str(Path(__file__).resolve()), "--config", str(Path(cfg["run"]) / "config.json"),\n                   "--task", "predict", "--part", part, "--gpu-slot", str(slot), *flags]\n            processes.append(subprocess.Popen(cmd))\n        codes = [p.wait() for p in processes]\n        if any(codes):\n            raise RuntimeError(f"Prediction subprocess failed: {codes}")\n    finally:\n        for p in processes:\n            if p.poll() is None:\n                p.terminate()\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    expected = {s["stem"] + ".npz" for s in manifest["splits"][part]}\n    assert {p.name for p in folder.glob("*.npz")} == expected\n    save_json(folder / "provenance.json", dict(key=key, models=paths, physical_images=len(expected), full_data_refit=full))\n    return folder\n\n\nFEATURES = ["log_area", "mean_probability", "max_probability", "q90_probability", "probability_std",\n    "boundary_probability", "fill", "aspect", "width_max", "width_mean", "gray_inside", "local_contrast",\n    "radius", "detector_confidence", "tta_support", "is_detection", "box_fill", "perimeter_ratio"]\n\n\ndef mask_features(box, mask, prob, edge, image, confidence=0., support=0., detection=False):\n    x0, y0, x1, y1 = box\n    p = prob[y0:y1, x0:x1][mask]\n    e = edge[y0:y1, x0:x1][mask]\n    gray = image[y0:y1, x0:x1, 1].astype(np.float32) / 255\n    distance = cv2.distanceTransform(np.pad(mask.astype(np.uint8), 1), cv2.DIST_L2, 3)[1:-1, 1:-1][mask]\n    ring = cv2.dilate(mask.astype(np.uint8), np.ones((9, 9), np.uint8)).astype(bool) & ~mask\n    contrast = float(gray[ring].mean() - gray[mask].mean()) if ring.any() else 0.\n    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    perimeter = sum(cv2.arcLength(c, True) for c in contours)\n    points = np.argwhere(mask)\n    height, width = mask.shape\n    area = int(mask.sum())\n    return [math.log1p(area), float(p.mean()), float(p.max()), float(np.quantile(p, .9)), float(p.std()),\n        float(e.mean()), area / (height * width), max(height, width) / max(min(height, width), 1),\n        float(distance.max()), float(distance.mean()), float(gray[mask].mean()), contrast,\n        float(np.hypot(points[:, 1].mean() + x0 - SIZE/2, points[:, 0].mean() + y0 - SIZE/2) / (SIZE/2)),\n        float(confidence), float(support), float(detection), area / (height * width),\n        float(perimeter / math.sqrt(max(area, 1)))]\n\n\ndef make_candidate(box, mask, prob, edge, image, confidence=0., support=0., detection=False):\n    return dict(rle=full_rle(box, mask), local_rle=encode(mask), box=list(map(int, box)), area=int(mask.sum()),\n        features=mask_features(box, mask, prob, edge, image, confidence, support, detection))\n\n\ndef candidate_sets(source, prediction_path, thresholds, maximum=512):\n    with np.load(prediction_path) as loaded:\n        prob, edge = loaded["prob"].astype(np.float32) / 255\n        boxes = loaded["boxes"]\n    image = cv2.imread(source["cache"])\n    result = {}\n    for threshold in thresholds:\n        dense = []\n        for box, mask in components(prob, threshold, 20, maximum=maximum):\n            dense.append(make_candidate(box, mask, prob, edge, image))\n        hybrid = []\n        for x0, y0, x1, y1, confidence, support in boxes:\n            x0, y0 = int(max(0, np.floor(x0) - 8)), int(max(0, np.floor(y0) - 8))\n            x1, y1 = int(min(SIZE, np.ceil(x1) + 8)), int(min(SIZE, np.ceil(y1) + 8))\n            if x1 <= x0 or y1 <= y0:\n                continue\n            # Preserve a detector\'s identity across disconnected mask pieces. Remove only tiny specks.\n            local = prob[y0:y1, x0:x1]\n            mask = np.zeros(local.shape, bool)\n            for (a, b, c, d), piece in components(local, threshold, 12):\n                mask[b:d, a:c] |= piece\n            if mask.sum() >= 20:\n                hybrid.append(make_candidate((x0, y0, x1, y1), mask, prob, edge, image,\n                    confidence=float(confidence), support=float(support), detection=True))\n        result[f"dense_{threshold}"] = dense\n        result[f"hybrid_{threshold}"] = hybrid + dense  # dense candidates can rescue detector misses\n    return result\n\n\ndef candidate_targets(source, candidates):\n    """Expected credited IoU over independent annotator records, using official >0.5."""\n    if not candidates:\n        return np.zeros(0, np.float32)\n    pred = [{**c["rle"], "counts": c["rle"]["counts"].encode()} for c in candidates]\n    result = np.zeros(len(pred), np.float64)\n    for view in source["views"]:\n        gt = [{**r, "counts": r["counts"].encode()} for r in view["rles"]]\n        if gt:\n            iou = np.asarray(mu.iou(gt, pred, [0] * len(pred)))\n            result += (iou * (iou > .5)).sum(0)\n    return (result / max(len(source["views"]), 1)).astype(np.float32)\n\n\ndef build_candidate_cache(cfg, part, folder, refresh=False):\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    out = Path(folder) / "candidates"\n    out.mkdir(exist_ok=True)\n    def process(source):\n        path = out / (source["stem"] + ".json")\n        if path.is_file() and not refresh:\n            return path\n        sets = candidate_sets(source, Path(folder) / (source["stem"] + ".npz"), cfg["pixel_thresholds"], cfg.get("max_dense_candidates", 512))\n        if part == "calibration":\n            for values in sets.values():\n                for c, y in zip(values, candidate_targets(source, values)):\n                    c["target"] = float(y)\n        save_json(path, sets)\n        return path\n    files = list(bounded_map(process, manifest["splits"][part], cfg["cpu_workers"]))\n    print(f"Candidate cache: {part}, {len(files)} sources", flush=True)\n    return out\n\n\ndef quality_estimator(seed):\n    from sklearn.ensemble import HistGradientBoostingRegressor\n    return HistGradientBoostingRegressor(max_iter=140, max_leaf_nodes=7, learning_rate=.05,\n        min_samples_leaf=30, l2_regularization=2., early_stopping=False, random_state=seed)\n\n\ndef fit_quality(cfg, candidate_folder):\n    """Date-grouped crossfitted scores for calibration, then fit all calibration for report/test.\n\n    Dense checkpoint selection has already used calibration; these are selection\n    scores, not unbiased CV. The separate report partition is never fitted/tuned.\n    """\n    import joblib\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    sources = manifest["splits"]["calibration"]\n    x, y, dates, indices = [], [], [], []\n    data = {}\n    for source in sources:\n        sets = load_json(Path(candidate_folder) / (source["stem"] + ".json"))\n        data[source["stem"]] = sets\n        for name, candidates in sets.items():\n            for j, c in enumerate(candidates):\n                x.append(c["features"])\n                y.append(c["target"])\n                dates.append(source["date"])\n                indices.append((source["stem"], name, j))\n    if not x:\n        raise RuntimeError("No calibration candidates. Train longer or inspect normalization and labels.")\n    x, y, dates = np.asarray(x, np.float32), np.asarray(y, np.float32), np.asarray(dates)\n    unique = np.array(sorted(set(dates)))\n    np.random.default_rng(cfg["seed"]).shuffle(unique)\n    folds = min(cfg["quality_folds"], len(unique))\n    if folds < 2:\n        raise RuntimeError("Quality crossfitting needs at least two independent calibration dates.")\n    assignment = {date: i % folds for i, date in enumerate(unique)}\n    oof = np.zeros(len(y), np.float32)\n    for fold in range(folds):\n        valid = np.array([assignment[d] == fold for d in dates])\n        model = quality_estimator(cfg["seed"] + fold)\n        model.fit(x[~valid], y[~valid])\n        oof[valid] = model.predict(x[valid])\n    for (stem, name, j), score in zip(indices, oof):\n        data[stem][name][j]["quality_oof"] = float(np.clip(score, 0, 1))\n    model = quality_estimator(cfg["seed"])\n    model.fit(x, y)\n    joblib.dump(model, Path(cfg["run"]) / "quality_model.joblib")\n    save_json(Path(cfg["run"]) / "quality_audit.json", dict(features=FEATURES,\n        training_partition="calibration only", crossfit_group="physical date", folds=folds,\n        candidate_rows=len(y), dates=len(unique), target="mean official credited IoU per annotation view",\n        crossfit_mae=float(np.abs(oof - y).mean()), report_used=False))\n    return model, data\n\n\ndef select_instances(candidates, recipe, quality=None, oof=False):\n    if not candidates:\n        return []\n    if recipe["quality"]:\n        scores = np.array([c["quality_oof"] for c in candidates]) if oof else np.clip(\n            quality.predict(np.asarray([c["features"] for c in candidates], np.float32)), 0, 1)\n    else:\n        scores = np.array([c["features"][1] for c in candidates])\n    order = np.argsort(-scores, kind="stable")\n    owned = np.zeros((SIZE, SIZE), bool)\n    accepted, rles, areas = [], [], []\n    adaptive = recipe.get("set", "").endswith("_adaptive")\n    for index in order:\n        c = candidates[index]\n        if scores[index] < recipe["cutoff"] or c["area"] < recipe["min_area"]:\n            continue\n        rle = {**c["rle"], "counts": c["rle"]["counts"].encode()}\n        if rles:\n            iou = np.asarray(mu.iou([rle], rles, [0] * len(rles)))[0]\n            if iou.max() > .5:\n                continue\n            if adaptive:\n                intersection = iou * (c["area"] + np.asarray(areas)) / (1 + iou)\n                if np.max(intersection / np.minimum(c["area"], areas)) > .8:\n                    continue\n        x0, y0, x1, y1 = c["box"]\n        local = c.get("_decoded")\n        if local is None:\n            local = decode(c["local_rle"])\n        mask = local & ~owned[y0:y1, x0:x1]\n        if mask.sum() < recipe["min_area"]:\n            continue\n        # Identity is retained after ownership, including disconnected pieces.\n        owned[y0:y1, x0:x1] |= mask\n        encoded = full_rle((x0, y0, x1, y1), mask)\n        accepted.append(encoded)\n        rles.append({**encoded, "counts": encoded["counts"].encode()})\n        areas.append(int(mask.sum()))\n    return accepted\n\n\ndef recipe_candidates(sets, name):\n    if name.endswith("_adaptive"):\n        family = name.split("_", 1)[0] + "_"\n        return [c for key, values in sets.items() if key.startswith(family) for c in values]\n    return sets[name]\n\n\ndef recipes(cfg):\n    result = []\n    for pixel in cfg["pixel_thresholds"]:\n        for family in ("dense", "hybrid"):\n            for quality in (False, True):\n                for cutoff in ([.15, .25, .35, .45] if quality else [.55, .7, .85]):\n                    for area in (50, 150):\n                        result.append(dict(set=f"{family}_{pixel}", quality=quality, cutoff=cutoff, min_area=area))\n    for family in ("dense", "hybrid"):\n        for cutoff in (.15, .25, .35, .45):\n            for area in (50, 150):\n                result.append(dict(set=f"{family}_adaptive", quality=True, cutoff=cutoff, min_area=area))\n    return result\n\n\ndef calibrate(cfg, candidate_folder):\n    model, data = fit_quality(cfg, candidate_folder)\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    sources = manifest["splits"]["calibration"]\n    candidates = recipes(cfg)\n    def score_source(source):\n        sets = data[source["stem"]]\n        for values in sets.values():\n            for c in values: c["_decoded"] = decode(c["local_rle"])\n        counts = np.zeros((len(candidates), 4), np.float64)\n        numbers = np.zeros(len(candidates), np.int64)\n        for i, recipe in enumerate(candidates):\n            pred = select_instances(recipe_candidates(sets, recipe["set"]), recipe, model, oof=True)\n            numbers[i] = len(pred)\n            for view in source["views"]:\n                counts[i] += metric_counts(view["rles"], pred)\n        for values in sets.values():\n            for c in values: c.pop("_decoded", None)\n        return counts, numbers\n    counts, numbers = np.zeros((len(candidates), 4)), np.zeros(len(candidates), np.int64)\n    for c, n in bounded_map(score_source, sources, cfg["cpu_workers"]):\n        counts += c\n        numbers += n\n    grid = [dict(**recipe, **summarize(c), physical_predictions=int(n))\n            for recipe, c, n in zip(candidates, counts, numbers)]\n    grid.sort(key=lambda x: x["PQ"], reverse=True)\n    chosen = {k: grid[0][k] for k in ("set", "quality", "cutoff", "min_area")}\n    save_json(Path(cfg["run"]) / "calibration_grid.json", grid)\n    save_json(Path(cfg["run"]) / "selected_pipeline.json", dict(recipe=chosen, calibration=grid[0],\n        score_type="selection score with date-crossfitted quality scores, not independent validation",\n        final_quality_fit="all calibration dates; report untouched", prediction_folder=str(candidate_folder.parent)))\n    print("Selected pipeline:", json.dumps(grid[0]), flush=True)\n    return chosen\n\n\ndef evaluate_report(cfg, candidate_folder, recipe):\n    import joblib\n    model = joblib.load(Path(cfg["run"]) / "quality_model.joblib")\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    totals, rows, cohorts = np.zeros(4, np.float64), [], defaultdict(lambda: np.zeros(4, np.float64))\n    for source in manifest["splits"]["report"]:\n        data = load_json(Path(candidate_folder) / (source["stem"] + ".json"))\n        pred = select_instances(recipe_candidates(data, recipe["set"]), recipe, model)\n        count = np.zeros(4, np.float64)\n        for view in source["views"]:\n            count += metric_counts(view["rles"], pred)\n        totals += count\n        cohort = "disk_fit" if source["photometry"]["geometry_ok"] else "fallback_normalized"\n        cohorts[cohort] += count\n        rows.append(dict(stem=source["stem"], date=source["date"], geometry_ok=source["photometry"]["geometry_ok"],\n            annotators=len(source["views"]), predictions=len(pred), **summarize(count)))\n    metrics = dict(**summarize(totals), physical_images=len(rows),\n        cohorts={k: summarize(v) for k, v in cohorts.items()},\n        note="Independent report; frozen recipe and calibration-fitted quality model. Before all-data refit.")\n    save_json(Path(cfg["run"]) / "report_metrics.json", metrics)\n    save_json(Path(cfg["run"]) / "report_by_image.json", rows)\n    print("Held-out report:", json.dumps(metrics, indent=2), flush=True)\n    return metrics\n\n\ndef export_submission(cfg, candidate_folder, recipe):\n    import joblib\n    model = joblib.load(Path(cfg["run"]) / "quality_model.joblib")\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    if cfg["mode"] == "full" and len(manifest["splits"]["test"]) != 180:\n        raise ValueError("Full export requires all 180 competition test images.")\n    name = "smoke_submission.csv" if cfg["mode"] == "smoke" else "submission.csv"\n    target = Path(cfg["run"]) / name\n    temporary = target.with_suffix(".tmp")\n    audit, identifiers = [], set()\n    with temporary.open("w", newline="", encoding="utf8") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["filament_id", "segmentation_rle"])\n        for source in manifest["splits"]["test"]:\n            data = load_json(Path(candidate_folder) / (source["stem"] + ".json"))\n            pred = select_instances(recipe_candidates(data, recipe["set"]), recipe, model)\n            union = np.zeros((SIZE, SIZE), bool)\n            for i, rle in enumerate(pred, 1):\n                mask = decode(rle)\n                assert mask.shape == (SIZE, SIZE) and mask.any() and not (union & mask).any()\n                assert encode(mask)["counts"] == rle["counts"]\n                union |= mask\n                identifier = f"{source[\'stem\']}_{i}"\n                assert identifier not in identifiers\n                identifiers.add(identifier)\n                writer.writerow([identifier, rle["counts"]])\n            audit.append(dict(image=source["name"], predictions=len(pred), geometry_ok=source["photometry"]["geometry_ok"]))\n    os.replace(temporary, target)\n    assert len(audit) == len(manifest["splits"]["test"])\n    if cfg["mode"] == "full" and len(audit) != 180:\n        raise ValueError(f"Expected 180 competition test images, found {len(audit)}. Do not submit this file.")\n    save_json(Path(cfg["run"]) / "submission_audit.json", dict(file=str(target), sha256=file_sha(target),\n        native_shape=[SIZE, SIZE], images=len(audit), masks=len(identifiers), zero_prediction_images=sum(r["predictions"] == 0 for r in audit),\n        per_image=audit, overlap_check=True, rle_roundtrip_check=True, recipe=recipe,\n        smoke=cfg["mode"] == "smoke", all_data_refit=cfg["refit_full"],\n        external_filament_ground_truth=False, training_annotation_sha=cfg["annotation_sha"]))\n    # Convenient Kaggle root artifact with the exact requested filename, only for full runs.\n    if cfg["mode"] == "full":\n        shutil.copy2(target, Path(cfg["output_root"]).resolve() / "submission.csv")\n    print(f"Wrote {target}: {len(audit)} images, {len(identifiers)} masks", flush=True)\n    return target\n\n\ndef preview(cfg, part, candidate_folder, recipe=None, limit=3):\n    import matplotlib.pyplot as plt\n    import joblib\n    manifest = load_json(Path(cfg["run"]) / "manifest.json")\n    sources = manifest["splits"][part][:limit]\n    model = joblib.load(Path(cfg["run"]) / "quality_model.joblib") if recipe else None\n    fig, axes = plt.subplots(len(sources), 3, figsize=(14, 4.5 * len(sources)), squeeze=False)\n    for row, source in zip(axes, sources):\n        gray = read_gray(source["image"])\n        row[0].imshow(gray, cmap="gray")\n        row[0].set_title(source["stem"])\n        with np.load(Path(candidate_folder).parent / (source["stem"] + ".npz")) as values:\n            row[1].imshow(values["prob"][0], cmap="magma", vmin=0, vmax=255)\n            row[1].set_title("Native ensemble probability")\n        row[2].imshow(gray, cmap="gray")\n        if recipe:\n            sets = load_json(Path(candidate_folder) / (source["stem"] + ".json"))\n            pred = select_instances(recipe_candidates(sets, recipe["set"]), recipe, model)\n            colors = plt.get_cmap("tab20")\n            for i, rle in enumerate(pred):\n                mask = decode(rle)\n                row[2].contour(mask, levels=[.5], colors=[colors(i % 20)], linewidths=.8)\n            row[2].set_title(f"{len(pred)} separate instances")\n        for axis in row: axis.axis("off")\n    fig.tight_layout()\n    path = Path(cfg["run"]) / f"{part}_predictions.png"\n    fig.savefig(path, dpi=150)\n    return fig\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--config", required=True)\n    p.add_argument("--task", choices=["dense", "detector", "predict"], required=True)\n    p.add_argument("--member", type=int, default=0)\n    p.add_argument("--full", action="store_true")\n    p.add_argument("--part", default="test")\n    p.add_argument("--gpu-slot", type=int, default=0)\n    args = p.parse_args()\n    if args.task == "dense": train_dense(args.config, args.member, args.full)\n    elif args.task == "detector": train_detector(args.config, args.full)\n    else: predict_worker(args.config, args.part, args.gpu_slot, args.full)\n\n\nif __name__ == "__main__":\n    main()\n\n'
RUNTIME_PATH.write_text(RUNTIME_SOURCE, encoding='utf8')
print('Embedded standalone runtime:', RUNTIME_PATH)

In [ ]:
import os
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[key] = '1'
import importlib
import torch
import pq50_runtime as pipeline
importlib.reload(pipeline)

CFG = pipeline.default_config()
CFG.update(
    mode='full',                 # 'smoke': three sources/partition, one short training epoch
    data_root='',                # auto-discover local/Kaggle competition train/ and test/
    output_root='/kaggle/working' if Path('/kaggle/working').exists() else './pq50_outputs',
    run_name='native_hybrid_v1',
    gpu_ids=[0, 1],               # GPU T4 x2. Explicitly use [0] for a local single-GPU check.
    cpu_workers=4,               # TOTAL: two training loader workers per GPU rank
    refit_full=True,              # frozen epoch counts, fresh training on all 707 images
)
# Resource controls, if needed: reduce dense_batch_per_gpu (2 -> 1) or
# inference_batch (2 -> 1). Keep crop/tile/detector sizes fixed for comparable results.
print('CUDA GPUs:', [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])
if not torch.cuda.is_available() or max(CFG['gpu_ids']) >= torch.cuda.device_count():
    raise RuntimeError('Select Kaggle GPU T4 x2, or explicitly configure the GPUs available on your machine.')
print('Mode:', CFG['mode'], '| GPUs:', CFG['gpu_ids'], '| CPU workers total:', CFG['cpu_workers'])

## Prepare shared inputs and independent annotation targets

The notebook measures compressed cache size and checks **incremental** disk needs.
Detector records hard-link the same image instead of creating full RGB copies.
Only best/last weights and compact prediction maps are retained. No automatic
recursive deletion is performed. Changing code, labels or configuration creates
a separate fingerprinted run, protecting older experiments.

In [ ]:
CFG, MANIFEST = pipeline.prepare(CFG)
RUN = Path(CFG['run'])
print('Run directory:', RUN)
for part, sources in MANIFEST['splits'].items():
    print(part, 'images=', len(sources), 'annotator records=', sum(len(s['views']) for s in sources),
          'geometry fallbacks=', sum(not s['photometry']['geometry_ok'] for s in sources))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

examples = MANIFEST['splits']['train'][:3]
fig, axes = plt.subplots(len(examples), 3, figsize=(13, 4 * len(examples)), squeeze=False)
for row, source in zip(axes, examples):
    row[0].imshow(pipeline.read_gray(source['image']), cmap='gray')
    row[0].set_title(source['stem'])
    row[1].imshow(cv2.imread(source['cache'])[:, :, 1], cmap='gray')
    row[1].set_title('Normalized; geometry=' + str(source['photometry']['geometry_ok']))
    packed = cv2.imread(source['views'][0]['target'], 0)
    row[2].imshow((packed & 1) > 0, cmap='gray')
    row[2].set_title('One annotator foreground target')
    for axis in row:
        axis.axis('off')
fig.tight_layout()
fig.savefig(RUN / 'preprocessing_preview.png', dpi=130)
plt.show()

## Train detector and native mask models

Each training job launches an importable script with `torchrun`, avoiding notebook
DDP pickling problems. Jobs train sequentially; each uses both GPUs. Mask losses
combine boundary-weighted BCE, foreground Dice and an auxiliary contour loss.
Encoder BN statistics are frozen, decoder normalization uses GroupNorm, and EMA
weights are evaluated. The cosine schedule completes its planned decay.

Full-mode detector validation uses one image and zero additional loader workers;
training uses two loader workers per rank. Generic COCO/ImageNet initialization
is downloaded, with **no weights trained on external solar-filament labels**.

In [ ]:
pipeline.train_all(CFG, full=False)
print('Selection model checkpoints:', pipeline.model_paths(CFG, full=False))

In [ ]:
import pandas as pd
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for member in range(len(CFG['dense_models'])):
    history = pd.read_json(RUN / 'selection_models' / f'dense_{member}' / 'history.jsonl', lines=True)
    label = CFG['dense_models'][member]['encoder']
    axes[0].plot(history.epoch, history.loss, label=label)
    checked = history.dropna(subset=['fixed_rule_calibration_PQ'])
    axes[1].plot(checked.epoch, checked.fixed_rule_calibration_PQ, marker='o', label=label)
axes[0].set(xlabel='Epoch', ylabel='Training loss')
axes[1].set(xlabel='Epoch', ylabel='Fixed-rule calibration PQ (selection)')
for axis in axes:
    axis.legend()
    axis.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN / 'training_history.png', dpi=150)
plt.show()

## Calibrate on dates excluded from training

Inference averages four exact flip views from both U-Nets, with Hann-weighted tile
blending. YOLO boxes are fused across views while retaining separate proposals.
Masks from proposals can retain disconnected pieces as one instance. Every final
pixel belongs to at most one mask. Plain semantic components are a competing
pipeline and an optional detector-miss recovery source.

Candidate generation retains at most 512 semantic components per threshold, ranked
by mean probability when that guard is reached. This bounds work for an untrained
or failing model while retaining small confident components. Normal filament counts
are far below this guard; `max_dense_candidates` is configurable.

The quality model uses image-derived contrast, shape, uncertainty, contour strength,
detector confidence and TTA support. It has no filename/site/label metadata at
inference. Calibration-only targets are the mean credited IoU over annotation
views. Date-grouped crossfitting prevents scoring each calibration candidate with
a quality model fitted on that date. The report will use a model fitted on all
calibration dates and the frozen recipe.

In [ ]:
CALIBRATION_MAPS = pipeline.predict_part(CFG, 'calibration', full=False)
CALIBRATION_CANDIDATES = pipeline.build_candidate_cache(CFG, 'calibration', CALIBRATION_MAPS)
RECIPE = pipeline.calibrate(CFG, CALIBRATION_CANDIDATES)
display(pd.DataFrame(pipeline.load_json(RUN / 'calibration_grid.json')).head(12))

## Evaluate the frozen pipeline once on the report partition

The implementation follows the organizer's strict **IoU > 0.5** overlap rule and
pools counts across **individual annotator-image records**. It does not score a
merged union or average per-image PQ. Keep this result as the honest pre-refit
estimate. Do not tune repeatedly against this set or assume it equals public PQ.

In [ ]:
REPORT_MAPS = pipeline.predict_part(CFG, 'report', full=False)
REPORT_CANDIDATES = pipeline.build_candidate_cache(CFG, 'report', REPORT_MAPS)
REPORT = pipeline.evaluate_report(CFG, REPORT_CANDIDATES, RECIPE)
display(pd.DataFrame([REPORT]))
pipeline.preview(CFG, 'report', REPORT_CANDIDATES, RECIPE)
plt.show()

## Frozen full-data refit and submission.csv

When enabled, retrain on train + calibration + report annotations using the epoch
counts already selected during the earlier stage. Keep the quality model and
postprocessing frozen. This uses all legal training data but has no remaining
independent local score; the saved report belongs to the previous models.
If semantic-only masks win calibration, skip the unused detector's full-data retrain.

The final CSV uses `filament_id,segmentation_rle`, native 2048² column-major COCO
compressed RLE counts, original filename stems and one-based instance counters.
Empty images are logged without invented masks. The exporter checks identifiers,
all test-image coverage, mask dimensions, RLE round trips and pixel non-overlap.

In [ ]:
if CFG['refit_full']:
    pipeline.train_all(CFG, full=True)
TEST_MAPS = pipeline.predict_part(CFG, 'test', full=CFG['refit_full'])
TEST_CANDIDATES = pipeline.build_candidate_cache(CFG, 'test', TEST_MAPS)
SUBMISSION = pipeline.export_submission(CFG, TEST_CANDIDATES, RECIPE)
print('Submission:', SUBMISSION)
display(pd.read_csv(SUBMISSION).head())
display(pipeline.load_json(RUN / 'submission_audit.json'))

In [ ]:
pipeline.preview(CFG, 'test', TEST_CANDIDATES, RECIPE)
plt.show()
from IPython.display import FileLink, display
display(FileLink(str(SUBMISSION)))
print('Full runs also copy submission.csv to:', Path(CFG['output_root']).resolve() / 'submission.csv')
print('Review report_metrics.json, submission_audit.json and the previews before uploading.')

## Research, rules and limits

The quantitative metric changed to PQ in August 2026. Old Dice leaderboard numbers
are not comparable. The organizer permits computational approaches but prohibits
other ground-truth metadata for training. This notebook uses only the competition
training annotations and generic COCO/ImageNet initialization. It does not retrieve
the public MAGFiLO annotation release, which overlaps competition test observations.

Primary sources reviewed:

- [Competition evaluation and data restrictions](https://www.kaggle.com/competitions/filament-segmentation-2026/overview/important-dates)
- [Organizer self-evaluation](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook)
- [Amey Pawar's detector/native-U-Net hybrid](https://github.com/ameyypawar/filament-seg): author-reported 0.4214 crossfit / 0.37 public; supports complementary proposals and masks, not a 0.50 promise.
- [HeShen's detection/refinement studies](https://github.com/HeShen-1/filament-segmentation-2026): author-reported 0.3593 five-fold / 0.35 public; documents correlated fusion errors and non-transferring training-length gains. Human agreement is not a mathematical model ceiling.
- [Artem Shelamanov's pipeline account](https://www.shelamanov.com/portfolio/solar): author-reported 0.40 public with native ROI masks and learned quality/fragment handling; code was unavailable when reviewed.
- [SMP U-Net documentation](https://smp.readthedocs.io/en/latest/models.html), [timm encoder interface](https://smp.readthedocs.io/en/latest/encoders_timm.html)
- [PyTorch DDP](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html), [Ultralytics training](https://docs.ultralytics.com/modes/train/)

This is a new implementation informed by those designs, not downloaded competition
weights or copied submissions. Dependencies retain their own licenses, including
Ultralytics AGPL-3.0. No independently reproduced rule-compliant 0.50 public recipe
was established by this research. Mask/proposal fusion and full-data refitting may
help or hurt; the notebook records evidence rather than manufacturing a score.

The accompanying workspace research folder contains the full training-label audit
and independent analysis of `Best/results.zip`. Smoke checks validate mechanics;
they do not establish model quality or validate two-T4 performance on a one-GPU PC.